# Annotated State Aggregation in LangGraph

This notebook demonstrates how `results: Annotated[list[dict], operator.add]` merges updates from multiple nodes into one shared state field.

It reuses shared helpers from `agent_patterns_common.py` and shared schema from `agent_pattern_states.py`.

## Architecture

```mermaid
graph LR
    S[Start] --> P[prepare]
    P --> A[node_customer]
    P --> B[node_product]
    P --> C[node_support]
    A --> K[coordinator]
    B --> K
    C --> K
    K --> E[End]
```

In [ ]:
import asyncio
import json

from langgraph.graph import END, START, StateGraph

from agent_pattern_states import AggregationState
from agent_patterns_common import log_event, render_state_graph

In [ ]:
async def prepare(state: AggregationState) -> dict:
    log_event("info", "node_start", node="prepare")
    return {"results": [], "history": ["prepare initialized fan-out"]}


async def node_customer(state: AggregationState) -> dict:
    await asyncio.sleep(0.2)
    payload = {"source": "customer", "value": "Customer profile loaded"}
    log_event("info", "node_complete", node="node_customer", payload=payload)
    return {
        "results": [payload],
        "history": ["customer data added"]
    }


async def node_product(state: AggregationState) -> dict:
    await asyncio.sleep(0.2)
    payload = {"source": "product", "value": "Product usage loaded"}
    log_event("info", "node_complete", node="node_product", payload=payload)
    return {
        "results": [payload],
        "history": ["product data added"]
    }


async def node_support(state: AggregationState) -> dict:
    await asyncio.sleep(0.2)
    payload = {"source": "support", "value": "Recent tickets loaded"}
    log_event("info", "node_complete", node="node_support", payload=payload)
    return {
        "results": [payload],
        "history": ["support data added"]
    }


async def coordinator(state: AggregationState) -> dict:
    ordered = sorted(state["results"], key=lambda item: item["source"])
    lines = [f"- {item['source']}: {item['value']}" for item in ordered]
    summary = "\n".join(lines)
    log_event(
        "info",
        "coordinator_merge_complete",
        merged_count=len(ordered),
        history_count=len(state["history"]),
    )
    return {
        "results": ordered,
        "final_answer": summary,
        "history": ["coordinator produced final answer"]
    }


def build_graph():
    builder = StateGraph(AggregationState)

    builder.add_node("prepare", prepare)
    builder.add_node("node_customer", node_customer)
    builder.add_node("node_product", node_product)
    builder.add_node("node_support", node_support)
    builder.add_node("coordinator", coordinator)

    builder.add_edge(START, "prepare")
    builder.add_edge("prepare", "node_customer")
    builder.add_edge("prepare", "node_product")
    builder.add_edge("prepare", "node_support")
    builder.add_edge("node_customer", "coordinator")
    builder.add_edge("node_product", "coordinator")
    builder.add_edge("node_support", "coordinator")
    builder.add_edge("coordinator", END)

    return builder.compile()


graph = build_graph()
render_state_graph(graph, title="Annotated Aggregation Graph")

In [ ]:
result = await graph.ainvoke(
    {
        "user_request": "Build a customer 360 summary",
        "results": [],
        "history": [],
        "final_answer": ""
    }
)

print("Merged results count:", len(result["results"]))
print("History events:", len(result["history"]))
print(result["final_answer"])

result

## Why this works

Because `results` is typed as `Annotated[list[dict], operator.add]` in `AggregationState`, each node contributes a list fragment and LangGraph merges them instead of overwriting earlier updates.

The same pattern is also used for `history` so parallel node traces are retained.